In [1]:
import sys
if '/disks/cosmodm/vdvuurst' not in sys.path:
    sys.path.append('/disks/cosmodm/vdvuurst')

import numpy as np
import h5py
from matplotlib import pyplot as plt
import os
from functional_forms import *
import ONEHALO
from importlib import reload
from functions import *
from tqdm import tqdm

from warnings import filterwarnings
# this might be a liiiitle dangerous but cleans up the output by a lot. that's because we often see invalid values in log10, but that's ok
filterwarnings('ignore',category = RuntimeWarning)

# joint_test = ONEHALO.ONEHALO_joint_fitter()
param_info = ONEHALO.param_info

In [6]:
use_full_set = False
MADD_test = ONEHALO.ONEHALO_MADD_fitter()
method = 'Nelder-Mead'

def get_init_DG_from_combi_nr(combi_nr, init_path = f'/disks/cosmodm/vdvuurst/data/onehalo_MADD_initial_conditions/{method}'):
    func_combi = all_combis[combi_nr - 1] if use_full_set else combi_subsample[subsample_num_to_idx[combi_nr]]
    n_params_r, n_params_m, ntot = param_info(func_combi)

    init_guess, _ = np.load(f'{init_path}/function_combi_{combi_nr}.npy')

    split_params = MADD_test.split_parameters(init_guess, n_params_m)
    DG = MADD_test.get_double_gauss_parameters(split_params, func_combi, n_params_r)
    return DG

def output_init_conditions(method = method):
    init_path = f'/disks/cosmodm/vdvuurst/data/onehalo_MADD_initial_conditions/{method}'
    bad_cnrs = []
    iterable = enumerate(tqdm(combi_numbers)) if use_full_set else enumerate(tqdm(combi_subsamples_numbers))
    for i, cnr in iterable:
        already_added = False

        sigma_1_init, sigma_2_init, _ = get_init_DG_from_combi_nr(cnr, init_path = init_path)
    
        if np.all(sigma_1_init == 2000.) or np.all(sigma_1_init == 1.):
            if cnr not in bad_cnrs:
                bad_cnrs.append(cnr)
            already_added = True
        
        elif np.all(sigma_2_init == 2000.) or np.all(sigma_2_init == 1.):
            if not already_added and cnr not in bad_cnrs:
                bad_cnrs.append(cnr)

    return np.array(bad_cnrs)

def verify_initials():
    print('Verifying initial conditions...')
    bad_cnrs = output_init_conditions(method).astype(int)
    cnrs_to_check = combi_numbers if use_full_set else combi_subsamples_numbers
    # good_cnrs = np.delete(cnrs_to_check, np.array(bad_cnrs) - 1) if use_full_set else np.delete(cnrs_to_check, [subsample_num_to_idx[bcnr] for bcnr in bad_cnrs])

    print(f'We find {bad_cnrs.shape[0]} bad combinations and {cnrs_to_check.size - bad_cnrs.shape[0]} good ones. ', end ='')

    return bad_cnrs

# print('Verifying intial conditions...')/
bad_cnrs = verify_initials()
# cnrs_to_check = combi_numbers if use_full_set else combi_subsamples_numbers
# # good_cnrs = np.delete(cnrs_to_check, np.array(bad_cnrs) - 1) # combi number -1 is an index, delete the bad ones and you're left with the good

# print(f'We find {bad_cnrs.shape[0]} bad combinations and {good_cnrs.shape[0]} good ones. Total is {cnrs_to_check.size}.')


Verifying initial conditions...


100%|██████████| 2624/2624 [00:10<00:00, 248.10it/s]

We find 1189 bad combinations and 1435 good ones. 

In [10]:
bad_cnrs, combi_subsamples_numbers, 1494 in bad_cnrs

(array([ 2517, 22898,  3994, ...,  4470,   181, 19717], shape=(1189,)),
 array([12098,  4345, 19001, ...,  1494,   470, 19717], shape=(2624,)),
 False)

In [11]:
get_init_DG_from_combi_nr(1494)

array([[2.95590942e+02, 2.26576935e+02, 3.60964417e+02, ...,
        2.83410736e+02, 3.33555511e+02, 2.34061356e+02],
       [9.52656479e+01, 1.00000000e+00, 1.43069397e+02, ...,
        1.66128784e+02, 1.80878479e+02, 1.72851807e+02],
       [2.44744297e-04, 1.75967711e-04, 3.48647620e-04, ...,
        5.03826886e-04, 1.35073450e-03, 6.92228205e-04]], shape=(3, 34447))

In [ ]:
bad_cnrs

array([19458, 22117, 22400,  6332, 22119, 13774, 20359,  1900, 13993,
       12162,  5746, 24191, 12250, 25552, 17020, 16539, 21501, 16049,
       18640,   729, 23655, 25716, 15266,  4778, 24830, 10249, 14099,
       22678, 21576,  8572,  4453, 10493, 17160,  8121,  5540, 23940,
       26072,  9008, 22913, 17365,  2569,  3914, 23945,  5057, 23079,
       18943, 25447, 12186, 14244,  1410, 18344,  5700,  8131, 20225,
        2428, 16991, 13987, 11412,  5760,  7988,  8681, 24106, 16965,
       12689,  2490, 23389,  3988, 21043, 25877, 21473, 14542,   664,
        2492, 20933,  3441, 17676,  3305,  1061,   191,  7023, 13169,
        5388, 18142,  5893,  2912, 17569,  1817, 15935, 15126, 21519,
       13809, 12384,  5424, 19627,  9475, 25317, 13185, 16922,  7575,
        7778,  8209, 18238, 21557,   855, 17413, 19175, 13068, 18033,
       21244, 23050,  5443,  6800,  8755, 12207, 15637, 20667, 13404,
       10806, 11262, 21573,   203, 25069,  7344,  4736, 21278, 25330,
        2434, 18092,

In [ ]:
np.load(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_1900.npy')[0], get_init_DG_from_combi_nr(1900)


(array([-6.65508864e+00,  7.95398235e-01, -4.48477649e-02, -3.20752999e+00,
         1.27548960e+01, -1.00231420e+00, -2.50089848e+00,  2.93739191e+00,
        -1.17193268e+00,  4.47295877e+01, -4.97687948e+00, -3.90686768e+00,
        -1.39544741e+00,  4.33330466e+00, -4.33438288e+00, -3.79015926e+00,
        -1.17344099e+00,  9.84815762e-02, -2.06587635e+03,  8.71560486e-05,
         6.14976716e-05,  4.84987913e-05,  0.00000000e+00]),
 array([[6.05983887e+02, 1.36547394e+02, 1.28205612e+02, ...,
         5.08264618e+02, 2.87686035e+02, 8.21671448e+02],
        [1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ...,
         1.00000000e+00, 1.00000000e+00, 1.00000000e+00],
        [9.49228823e-04, 6.49296038e-04, 2.30785413e-03, ...,
         4.74016910e-04, 1.62899413e-03, 1.01116335e-03]], shape=(3, 34447)))

In [ ]:
np.load(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_{2}.npy')[0], get_init_DG_from_combi_nr(2)


[array([4.46424964, 0.74687538]), array([-7.43305824, -8.04247911]), array([20.63932932,  2.03459796]), array([ -3.34452537, 382.97961185]), array([-3.24261891, -3.16248719]), array([  -7.09169526, 1096.74741239]), array([6.61347885e-05, 1.92010724e-03]), array([0.        , 0.03109657]), array([0.00024426, 0.00245747, 0.00034332])]


(array([ 4.46424964e+00,  7.46875383e-01, -7.43305824e+00, -8.04247911e+00,
         2.06393293e+01,  2.03459796e+00, -3.34452537e+00,  3.82979612e+02,
        -3.24261891e+00, -3.16248719e+00, -7.09169526e+00,  1.09674741e+03,
         6.61347885e-05,  1.92010724e-03,  0.00000000e+00,  3.10965681e-02,
         2.44255107e-04,  2.45747362e-03,  3.43316388e-04]),
 array([[1.05656128e+03, 1.07299805e+03, 1.07579932e+03, ...,
         1.04713794e+03, 1.06729407e+03, 1.05371680e+03],
        [4.11703735e+02, 3.97376221e+02, 3.81368622e+02, ...,
         4.62377411e+02, 3.88669312e+02, 4.13960297e+02],
        [9.82560383e-01, 9.89915775e-01, 9.89067235e-01, ...,
         9.84342938e-01, 9.85457664e-01, 9.81081165e-01]], shape=(3, 34447)))

In [ ]:
# sigma_1_change_step = int(len(all_names) / len(sigma_1_funcs_names))
# sigma_2_change_step = int(len(all_names) / len(sigma_2_funcs_names))

# no_sigma_1_params_replacement = []
# no_sigma_2_params_replacement = []
# used_good_nrs = []
# for n in range(len(sigma_1_funcs_names)):
#     good_cnr_to_use = good_cnrs[good_cnrs > (n*sigma_1_change_step + 1)][0]
#     used_good_nrs.append(good_cnr_to_use)

#     param_values, MCMC_steps = np.load(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_{good_cnr_to_use}.npy')
#     n_params_r, n_params_m, _ = ONEHALO.param_info(all_combis[good_cnr_to_use - 1])

#     no_sigma_1_params = sum(n_params_m[:n_params_r[0]])
#     no_sigma_2_params = sum(n_params_m[n_params_r[0]:n_params_r[1]])
    
#     no_sigma_1_params_replacement.append(no_sigma_1_params)
#     no_sigma_2_params_replacement.append(no_sigma_2_params)


# for i, cnr in tqdm(enumerate(combi_numbers), total = 9216):
#     if cnr in good_cnrs:
#         continue
#     else: 
#         replacement_idx = i // sigma_1_change_step
#         replacement_values, replacement_MCMC_steps = np.load(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_{used_good_nrs[replacement_idx]}.npy')

#         old_values, old_MCMC_steps = np.load(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_{cnr}.npy')

#         new_values, new_MCMC_steps = old_values.copy(), old_MCMC_steps.copy()
#         new_values[:no_sigma_1_params], new_MCMC_steps[:no_sigma_1_params] = replacement_values[:no_sigma_1_params], replacement_MCMC_steps[:no_sigma_1_params]

#         func_combi = all_combis[cnr - 1]
#         func_combi_names = all_names[cnr - 1]
#         n_params_r, n_params_m, ntot = ONEHALO.param_info(func_combi)
#         split_params = joint_test.split_parameters(new_values, n_params_m)
#         DG = joint_test.get_double_gauss_parameters(split_params, func_combi, n_params_r)
#         if np.all(DG[0] == 2000.): #i.e. the problem was actually sigma 2
#             new_values, new_MCMC_steps = old_values.copy(), old_MCMC_steps.copy()

#             # THIS IS WRONG: it takes the functional form from a good combi that has the same SIGMA_1 functional form, not sigma 2. first you need to find that one (SOMEHOW??????????????)
#             new_values, new_MCMC_steps = replacement_values[no_sigma_1_params:no_sigma_1_params+no_sigma_2_params], replacement_MCMC_steps[no_sigma_1_params:no_sigma_1_params+no_sigma_2_params]

#         #overwrite
#         np.save(f'/disks/cosmodm/vdvuurst/data/onehalo_joint_initial_conditions/Nelder-Mead/function_combi_{cnr}.npy', np.vstack((new_values, new_MCMC_steps)))

# print(no_sigma_1_params_replacement)

  6%|▌         | 542/9216 [00:02<00:30, 280.17it/s]

100%|██████████| 9216/9216 [00:33<00:00, 275.06it/s]

[8, 9, 9, 10, 10, 10, 11, 11, 11, 11, 12, 12, 12, 12, 12, 10, 11, 11, 12, 12, 12, 13, 13, 13, 13, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15]


In [ ]:
all_names[::16, 1]

array([['linear', ('linear', 'linear')],
       ['linear', ('linear', 'parabola')],
       ['linear', ('linear', 'exponential')],
       ...,
       ['parabola', ('parabola', 'parabola', 'exponential')],
       ['parabola', ('parabola', 'exponential', 'exponential')],
       ['parabola', ('exponential', 'exponential', 'exponential')]],
      shape=(576, 2), dtype=object)

In [ ]:
for name in all_names[::(16), 1]:
    print(name)

['linear' ('linear', 'linear')]
['linear' ('linear', 'parabola')]
['linear' ('linear', 'exponential')]
['linear' ('parabola', 'parabola')]
['linear' ('parabola', 'exponential')]
['linear' ('exponential', 'exponential')]
['parabola' ('linear', 'linear', 'linear')]
['parabola' ('linear', 'linear', 'parabola')]
['parabola' ('linear', 'linear', 'exponential')]
['parabola' ('linear', 'parabola', 'parabola')]
['parabola' ('linear', 'parabola', 'exponential')]
['parabola' ('linear', 'exponential', 'exponential')]
['parabola' ('parabola', 'parabola', 'parabola')]
['parabola' ('parabola', 'parabola', 'exponential')]
['parabola' ('parabola', 'exponential', 'exponential')]
['parabola' ('exponential', 'exponential', 'exponential')]
['linear' ('linear', 'linear')]
['linear' ('linear', 'parabola')]
['linear' ('linear', 'exponential')]
['linear' ('parabola', 'parabola')]
['linear' ('parabola', 'exponential')]
['linear' ('exponential', 'exponential')]
['parabola' ('linear', 'linear', 'linear')]
['para